# Week 2 — Data analysis with AI as pair programmer
**ESE · AI for Business and FinTech · 28 September 2026**

Goal of the session: acquire, clean, join and chart financial data with the assistant writing most of the code — while you catch its silent errors. By the end you will have a reproducible pipeline that joins market prices with an external macro series and an on-chain series.

In [ ]:
!pip -q install yfinance

In [ ]:
# --- course helper: price loader with fallbacks (run this cell once per session) ---
import warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")

def load_prices(tickers, start="2022-01-01", end=None, cache_dir="data"):
    """Daily close prices, one column per ticker.
    1) try yfinance (live)  2) try a CSV snapshot in data/  3) synthetic random walk (pipeline test only)."""
    import os
    os.makedirs(cache_dir, exist_ok=True)
    key = "_".join(t.replace("^", "") for t in tickers)
    snap = os.path.join(cache_dir, f"prices_{key}.csv")
    try:
        import yfinance as yf
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        px = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw[["Close"]].rename(columns={"Close": tickers[0]})
        px = px.dropna(how="all")
        if len(px) < 50:
            raise RuntimeError("empty download")
        px.to_csv(snap)
        print(f"[live] {px.shape[0]} rows from yfinance; snapshot saved to {snap}")
        return px
    except Exception as e:
        print(f"[warn] yfinance failed ({type(e).__name__}); trying snapshot")
    if os.path.exists(snap):
        px = pd.read_csv(snap, index_col=0, parse_dates=True)
        print(f"[snapshot] {px.shape[0]} rows from {snap}")
        return px
    print("[SYNTHETIC] no network and no snapshot: generating a random walk. Numbers below are NOT real.")
    rng = np.random.default_rng(0)
    idx = pd.bdate_range(start, end or pd.Timestamp.today().normalize())
    px = pd.DataFrame({t: 100 * np.exp(np.cumsum(rng.normal(0.0004, 0.02 if "USD" in t else 0.011, len(idx))))
                       for t in tickers}, index=idx)
    return px

## Block A — pandas by doing

In [ ]:
px = load_prices(["BTC-USD", "SPY", "AAPL"], start="2022-01-01")
print(px.index.min().date(), "→", px.index.max().date(), "|", px.shape)
px.head(3)

A DataFrame is a spreadsheet with an index. The four operations you will use 90% of the time:

| Operation | Excel analogue | pandas |
|---|---|---|
| filter rows | AutoFilter | `df[df["col"] > 0]` |
| new column | formula column | `df["ret"] = df["px"].pct_change()` |
| aggregate | PivotTable | `df.groupby(...).agg(...)` / `df.resample("W").last()` |
| join | VLOOKUP / Power Query merge | `df.join(other)` / `pd.merge(...)` |

In [ ]:
# Missing values: BTC trades 7 days a week, SPY and AAPL 5. Look at a week that contains a weekend.
px.loc["2024-01-05":"2024-01-09"]

**🔍 CHECK.** `load_prices` already dropped rows where *all* columns were missing, but weekend rows survive because BTC has a value. Before computing anything across assets you must decide: drop weekends (equity calendar), or fill equities forward (crypto calendar)? Neither is "right" — but choosing silently is wrong. Which one does `pct_change()` implicitly choose if you do nothing?

In [ ]:
# Two explicit choices, side by side
px_eq = px.dropna()                       # equity calendar: only days where everything traded
px_cr = px.ffill()                        # crypto calendar: carry equities over the weekend
print("equity calendar rows:", len(px_eq), "| crypto calendar rows:", len(px_cr))

rets_eq = px_eq.pct_change().dropna()
rets_cr = px_cr.pct_change().dropna()
print("\nBTC daily vol — equity calendar: %.4f | crypto calendar: %.4f" % (rets_eq["BTC-USD"].std(), rets_cr["BTC-USD"].std()))
print("BTC mean Monday return on equity calendar (Fri→Mon, 3 days of moves): %.4f" % rets_eq["BTC-USD"][rets_eq.index.dayofweek == 0].mean())

In [ ]:
# Resampling: weekly prices (last obs of the week), and weekly returns
wk = px_eq.resample("W-FRI").last()
wk_rets = wk.pct_change().dropna()
wk_rets.tail()

## Block B — Join with external series (macro + on-chain)

This is where analysis becomes interesting: internal or market data alone rarely answers a business question. We add:
- **US 10-year Treasury yield** (FRED series `DGS10`), a proxy for the risk-free rate / macro regime;
- **Bitcoin unique active addresses** (blockchain.com public chart API), an on-chain adoption proxy.

Both are free and need no key. Both fetches have a fallback so the notebook still runs offline.

In [ ]:
import requests, io, os

def load_fred(series="DGS10", cache_dir="data"):
    os.makedirs(cache_dir, exist_ok=True); snap = f"{cache_dir}/fred_{series}.csv"
    try:
        url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series}"
        s = pd.read_csv(io.StringIO(requests.get(url, timeout=20).text), index_col=0, parse_dates=True)[series]
        s = pd.to_numeric(s, errors="coerce").rename(series); s.to_csv(snap); print("[live] FRED", series); return s
    except Exception as e:
        print("[warn] FRED failed:", type(e).__name__)
    if os.path.exists(snap):
        print("[snapshot] FRED"); return pd.read_csv(snap, index_col=0, parse_dates=True).iloc[:, 0].rename(series)
    print("[SYNTHETIC] FRED"); idx = pd.bdate_range("2022-01-01", pd.Timestamp.today().normalize())
    return pd.Series(3 + np.cumsum(np.random.default_rng(1).normal(0, 0.03, len(idx))), idx, name=series)

def load_btc_addresses(cache_dir="data"):
    os.makedirs(cache_dir, exist_ok=True); snap = f"{cache_dir}/btc_active_addresses.csv"
    try:
        url = "https://api.blockchain.info/charts/n-unique-addresses?timespan=5years&format=json&sampled=false"
        js = requests.get(url, timeout=20).json()["values"]
        s = pd.Series({pd.to_datetime(v["x"], unit="s"): v["y"] for v in js}, name="active_addresses")
        s.to_csv(snap); print("[live] blockchain.com"); return s
    except Exception as e:
        print("[warn] blockchain.com failed:", type(e).__name__)
    if os.path.exists(snap):
        print("[snapshot] blockchain.com"); return pd.read_csv(snap, index_col=0, parse_dates=True).iloc[:, 0].rename("active_addresses")
    print("[SYNTHETIC] addresses"); idx = pd.date_range("2022-01-01", pd.Timestamp.today().normalize())
    return pd.Series(7e5 + 1e5 * np.sin(np.arange(len(idx)) / 60) + np.random.default_rng(2).normal(0, 5e4, len(idx)), idx, name="active_addresses")

dgs10 = load_fred("DGS10")
addr = load_btc_addresses()
print(dgs10.tail(3)); print(addr.tail(3))

### 🔍 CHECK — the assistant's join
Asked to "merge the price data with the yield and the on-chain series", the assistant wrote the cell below. It runs. Look at the row count and the date range of the result before and after, then explain what was lost and why.

In [ ]:
# --- as produced by the assistant (do not trust) ---
merged_bad = pd.merge(px, dgs10, left_index=True, right_index=True).merge(addr, left_index=True, right_index=True)
print("rows before:", len(px), "| rows after:", len(merged_bad))
print(merged_bad.index.min().date(), "→", merged_bad.index.max().date())
merged_bad.isna().sum()

<details><summary>What went wrong</summary>

`pd.merge` defaults to an **inner join**: only dates present in *all three* series survive. FRED has no weekends and has `NaN` on US holidays (which survive as NaN — check the `isna()` output); the on-chain series is daily. The result silently dropped weekends and is on an equity calendar without anyone deciding that. Also: FRED yields are in **percent** (4.25 = 4.25%), prices in dollars, addresses in counts — the assistant will happily correlate them without saying so. Units are your job.
</details>

In [ ]:
# Explicit version: choose the calendar (equity days), left-join, forward-fill macro, state units.
base = px.dropna()                                    # equity calendar
df = base.join(dgs10.ffill(), how="left").join(addr, how="left")
df["DGS10"] = df["DGS10"] / 100                       # now a decimal rate
df = df.rename(columns={"DGS10": "us10y"})
df["active_addresses"] = df["active_addresses"].ffill()
print(len(base), len(df)); df.tail()

## Block C — Charts that answer a question, and rolling statistics

In [ ]:
import matplotlib.pyplot as plt
rets = df[["BTC-USD", "SPY", "AAPL"]].pct_change()
roll_corr = rets["BTC-USD"].rolling(60).corr(rets["SPY"])

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
roll_corr.plot(ax=ax[0], title="60-day rolling correlation, BTC vs SPY")
ax[0].axhline(0, color="k", lw=0.5)
df["us10y"].plot(ax=ax[1], title="US 10y yield (decimal)")
events = {"2024-01-10": "spot ETF approval", "2024-04-19": "halving"}
for d, label in events.items():
    if pd.Timestamp(d) in df.index:
        for a in ax: a.axvline(pd.Timestamp(d), color="grey", ls="--", lw=0.8)
        ax[0].annotate(label, (pd.Timestamp(d), roll_corr.max() * 0.9), fontsize=8, rotation=90)
plt.tight_layout(); plt.show()

**🔍 CHECK.** Does the correlation chart *answer a question*, or only show a line? Write the question it answers in one sentence. If you cannot, the chart is not finished. (Sanity check the data too: you should see BTC's correlation with equities fall after some events and rise in macro-stress periods. If the line looks like noise around zero everywhere, suspect an alignment problem.)

In [ ]:
# On-chain vs price: is address activity leading or lagging price? Cross-correlation at different lags (weekly data).
w = df[["BTC-USD", "active_addresses"]].resample("W-FRI").last().pct_change().dropna()
lags = range(-8, 9)
xc = pd.Series({k: w["BTC-USD"].corr(w["active_addresses"].shift(k)) for k in lags})
xc.plot(kind="bar", figsize=(9, 3), title="corr(BTC weekly return, address growth shifted by k weeks)  — k>0: addresses lead")
plt.show()

**🔍 CHECK.** Positive `k` means address growth from `k` weeks *ago* vs return *now*. If the biggest bar is at negative `k`, price leads activity, not the reverse — the opposite of the story "adoption drives price". Read the sign convention in `shift()` before you conclude anything. This is exactly the kind of check an assistant will not do for you.

## Block D — Reproducibility and privacy

In [ ]:
# 1) Snapshot the exact data you analysed, with the date, so the memo can be re-run.
import datetime as dt, os
os.makedirs("data", exist_ok=True)
stamp = dt.date.today().isoformat()
df.to_csv(f"data/week02_merged_{stamp}.csv")
print("saved", f"data/week02_merged_{stamp}.csv", df.shape)

# 2) Record the environment.
!pip freeze 2>/dev/null | grep -i -E "^(pandas|numpy|yfinance|matplotlib)=" > data/requirements_week02.txt; cat data/requirements_week02.txt

**Rules from here on:**
- every notebook that produces a number in a memo saves the data it used (`data/…_YYYY-MM-DD.csv`);
- random operations set a seed (`np.random.default_rng(0)`);
- nothing confidential goes into an AI tool: no client data, no personal data, no internal documents. Public market data is fine. If in doubt, anonymise or aggregate first.

**Homework brief** → see `week-02/homework.md`.